# src03 — Generación de Lenguaje Natural

**Entrada:** CSV de reglas generado por `src02` → `data/<NOMBRE_DATASET>_<metrica>_reglas.csv`  
**Salida:** Resumen en lenguaje natural → `data/<NOMBRE_DATASET>_<metrica>_resumen.txt`

El notebook agrupa las reglas de asociación difusas por consecuente y tipo temporal,  
y genera párrafos descriptivos en español mediante plantillas lingüísticas jerárquicas.

**Pipeline:** raw → fuzzificación (src01) → reglas (src02) → **resumen NL (src03)**

In [ ]:
# @title 1. Imports y configuración

USUARIO = "IZARO"  # @param ["IZARO","ENRIQUE"]
import os
import re
import textwrap
from itertools import groupby
from collections import defaultdict
import pandas as pd
from google.colab import drive
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

rutas = {
    "IZARO":   "/content/drive/MyDrive/05. PFG/2026 - TFG Izaro Fuzhi/Codigo",
    "ENRIQUE": "/content/drive/MyDrive/2-DOCENCIA/ALL Trabajos Fin de Estudios/2026 - TFG Izaro Fuzhi/Codigo",
}
DIRECTORIO = rutas[USUARIO]
drive.mount("/content/drive")
os.chdir(DIRECTORIO)
DIR_DATOS = "data"
print(f"Directorio de trabajo: {DIRECTORIO}")


In [ ]:
# @title 2. Parámetros

#SENSOR  = "6823"       # @param ["3600","3730","4301","6791","6822","6823"]
#METRICA = "ocupacion" # @param ["intensidad","ocupacion"]
NOMBRE_DATASET = "DailyDelhiClimateTrain_meantemp"
METRICA = "meantemp"

# Número mínimo de reglas por grupo para que se genere un párrafo conjunto
# (grupos con menos reglas se describen individualmente)
MIN_REGLAS_GRUPO = 2   # @param {"type":"integer"}

NOMBRE_DATASET = "DailyDelhiClimateTrain"
METRICA = "meantemp"
FICHERO_REGLAS  = f"data/{NOMBRE_DATASET}_{METRICA}_reglas.csv"
FICHERO_FUZZY   = f"data/{NOMBRE_DATASET}_{METRICA}_fuzzy.csv"
FICHERO_RESUMEN = f"data/{NOMBRE_DATASET}_{METRICA}_resumen.md"

df_reglas = pd.read_csv(FICHERO_REGLAS)
print(f"Reglas cargadas: {len(df_reglas)}")
print(df_reglas.head(10).to_string(index=False))


In [ ]:
import pandas as pd
import numpy as np
import re

# ── Detección de bloques temporales disponibles ──────────────────────────────
# Se infiere directamente de las columnas del CSV fuzzificado,
# que refleja exactamente qué generó src01.

# Leer columnas del fuzzy CSV (que es el que tiene todas las t_*) y la columna de tiempo
_df_fuzzy = pd.read_csv(FICHERO_FUZZY)
_cols_fuzzy = set(_df_fuzzy.columns)

# ───── Detección real basada en columnas existentes ─────
HAY_ANIOS      = any(re.match(r'^t_\d{4}$', c) for c in _cols_fuzzy)
HAY_MESES      = any(c in _cols_fuzzy for c in ["t_Ene","t_Feb","t_Marz"]) # Ajusta si añadiste más en src01
HAY_ESTACIONES = any(c in _cols_fuzzy for c in ["t_Primavera","t_Verano","t_Otonio","t_Invierno"])
HAY_QUINCENAS  = any(c in _cols_fuzzy for c in ["t_Q1mes","t_Q2mes"])
HAY_DIAS       = any(c in _cols_fuzzy for c in ["t_Lun","t_Mar","t_Mie","t_Jue","t_Vie","t_Sab","t_Dom"])
HAY_LABORABLES = any(c in _cols_fuzzy for c in ["t_Laborable","t_FinSemana"]) # Asegúrate de que coincida el nombre de src01
HAY_FRANJAS    = any(c in _cols_fuzzy for c in ["t_Madrugada","t_Mañana","t_Tarde","t_Noche"])
HAY_HORAS      = any(c.startswith("t_H") for c in _cols_fuzzy)
HAY_MINUTOS    = any(c.startswith("t_M") and c[2:].isdigit() and int(c[2:]) % 15 == 0 for c in _cols_fuzzy) # Cuartos
HAY_MIN_FINOS  = any(c.startswith("t_M") and c[2:].isdigit() and int(c[2:]) % 15 != 0 for c in _cols_fuzzy) # Minutos finos
HAY_FESTIVOS   = "t_Festivo" in _cols_fuzzy

# ───── Procesamiento de la Variable de Tiempo y Granularidad ─────
VAR_TIEMPO = 'date'
_granularidad_desc = "desconocida"
GRANULARIDAD_S = 3600.0  # Predeterminado por seguridad

if VAR_TIEMPO in _df_fuzzy.columns:
    _df_fuzzy[VAR_TIEMPO] = pd.to_datetime(_df_fuzzy[VAR_TIEMPO])
    t0 = _df_fuzzy[VAR_TIEMPO].min()
    _t_max = _df_fuzzy[VAR_TIEMPO].max()

    _df_fuzzy["segundos"] = (_df_fuzzy[VAR_TIEMPO] - t0).dt.total_seconds().astype(int)
    x     = _df_fuzzy["segundos"].to_numpy()
    x_max = int(_df_fuzzy["segundos"].max())
    anio_inicio = t0.year
    anio_fin    = _df_fuzzy[VAR_TIEMPO].max().year

    _diffs_tmp = _df_fuzzy[VAR_TIEMPO].diff().dt.total_seconds().dropna()
    if len(_diffs_tmp) > 0:
        GRANULARIDAD_S = float(_diffs_tmp.median())
    del _diffs_tmp
else:
    print(f"Advertencia: La columna '{VAR_TIEMPO}' no se encontró en el archivo fuzzy. Se usarán valores predeterminados.")
    _df_fuzzy["segundos"] = 0
    x = np.array([0])
    x_max = 0
    anio_inicio = 0
    anio_fin = 0
    t0 = pd.Timestamp('2000-01-01')
    _t_max = pd.Timestamp('2000-01-01')

# Métricas de cobertura calculadas en este dataset cargado
COBERTURA_S     = (_t_max - t0).total_seconds()
COBERTURA_DIAS  = COBERTURA_S / 86400
N_ANIOS_DISTINTOS = _df_fuzzy[VAR_TIEMPO].dt.year.nunique() if VAR_TIEMPO in _df_fuzzy.columns else 0
N_MESES_DISTINTOS = _df_fuzzy[VAR_TIEMPO].dt.to_period("M").nunique() if VAR_TIEMPO in _df_fuzzy.columns else 0

# Clasificación textual de la granularidad encontrada
if   GRANULARIDAD_S <= 60:     _nivel = "subminuto"
elif GRANULARIDAD_S <= 3600:   _nivel = "subhora"
elif GRANULARIDAD_S <= 86400:  _nivel = "subdia"
elif GRANULARIDAD_S <= 604800: _nivel = "subsemana"
else:                          _nivel = "grueso"

# Descripción legible para lógica interna posterior
if GRANULARIDAD_S >= 86400:   _granularidad_desc = "diaria"
elif GRANULARIDAD_S >= 3600:  _granularidad_desc = "horaria"
elif GRANULARIDAD_S >= 900:   _granularidad_desc = "de 15 minutos"
else:                         _granularidad_desc = f"de {int(GRANULARIDAD_S)}s"

# ───── Salida en Pantalla Consistente con SRC01 ─────
print("=" * 60)
print("DIAGNÓSTICO DEL DATASET FUZZIFICADO (SRC03)")
print("=" * 60)
print(f"  Granularidad mediana: {GRANULARIDAD_S:>10.1f} s   ({_nivel})")
print(f"  Cobertura temporal:   {COBERTURA_DIAS:>10.1f} días")
print(f"  Años distintos:       {N_ANIOS_DISTINTOS:>10d}")
print(f"  Meses distintos:      {N_MESES_DISTINTOS:>10d}")
print()
print("BLOQUES TEMPORALES DETECTADOS (DISPONIBLES)")
print("-" * 60)

for nombre, valor in [
    ("Años",                 HAY_ANIOS),
    ("Meses",                HAY_MESES),
    ("Estaciones",           HAY_ESTACIONES),
    ("Quincenas",            HAY_QUINCENAS),
    ("Días (Lun..Dom)",      HAY_DIAS),
    ("Laborable/FinSem",     HAY_LABORABLES),
    ("Franjas del día",      HAY_FRANJAS),
    ("Horas",                HAY_HORAS),
    ("Minutos (cuartos)",    HAY_MINUTOS),
    ("Minutos finos (<60s)", HAY_MIN_FINOS),
    ("Festivos",             HAY_FESTIVOS),
]:
    marca = "✓" if valor else "✗"
    print(f"  {marca} {nombre}")
print("=" * 60)

## 3. Diccionarios de traducción y jerarquía temporal

In [ ]:
# ---------------------------------------------------------------------------
# 3. Diccionarios de traducción y jerarquía temporal
# ---------------------------------------------------------------------------

# ── 3a. Etiquetas legibles para cada token temporal ────────────────────────
ETIQUETA_TEMPORAL = {
    # Años — se completan dinámicamente más abajo con los años del CSV
    **{f"t_{y}": f"el año {y}" for y in range(2000, 2051)},
    # Meses
    "t_Ene": "enero",    "t_Feb": "febrero",  "t_Marz": "marzo",
    "t_Abr": "abril",    "t_May": "mayo",     "t_Jun": "junio",
    "t_Jul": "julio",    "t_Ago": "agosto",   "t_Sep": "septiembre",
    "t_Oct": "octubre",  "t_Nov": "noviembre", "t_Dic": "diciembre",
    # Días de la semana
    "t_Lun": "los lunes",  "t_Mar": "los martes",  "t_Mie": "los miércoles",
    "t_Jue": "los jueves", "t_Vie": "los viernes",  "t_Sab": "los sábados",
    "t_Dom": "los domingos",
    # Horas (H00–H23) — se generan programáticamente más abajo
    **{f"t_H{h:02d}": f"las {h}h" for h in range(24)},
    # Franjas horarias
    "t_Madrugada": "la madrugada (0–6 h)",
    "t_Mañana":    "la mañana (7–13 h)",
    "t_Tarde":     "la tarde (14–20 h)",
    "t_Noche":     "la noche (21–23 h)",
    # Tipo de día
    "t_Laborable": "días laborables",
    "t_FinSemana": "fin de semana",
    # Quincenas
    "t_Q1mes": "la primera quincena del mes",
    "t_Q2mes": "la segunda quincena del mes",
    # Estaciones
    "t_Primavera": "primavera",
    "t_Verano":    "verano",
    "t_Otonio":    "otoño",
    "t_Invierno":  "invierno",
    # Festivos (nuevo — bloque GEN_FESTIVOS de src01)
    "t_Festivo":   "días festivos",
    # Minutos — cuartos de hora (nuevo — bloque GEN_MINUTOS de src01)
    "t_M00": "el primer cuarto de hora (minutos 0–14)",
    "t_M15": "el segundo cuarto de hora (minutos 15–29)",
    "t_M30": "el tercer cuarto de hora (minutos 30–44)",
    "t_M45": "el último cuarto de hora (minutos 45–59)",
}

# ── 3b. Etiquetas para las variables de métrica (consecuente) ─────────────
ETIQUETA_METRICA_COLOQUIAL = {
    "v_MuyBaja":     "muy baja",
    "v_Baja":        "baja",
    "v_Media":       "media",
    "v_Alta":        "alta",
    "v_MuyAlta":     "muy alta",
    "v_OutlierBajo": "excepcionalmente baja",  # sin "(outlier inferior)"
    "v_OutlierAlto": "excepcionalmente alta",  # sin "(outlier superior)"
}
ETIQUETA_METRICA_TECNICA = {
    "v_MuyBaja":     "muy baja",
    "v_Baja":        "baja",
    "v_Media":       "media",
    "v_Alta":        "alta",
    "v_MuyAlta":     "muy alta",
    "v_OutlierBajo": "excepcionalmente baja (outlier inferior)",
    "v_OutlierAlto": "excepcionalmente alta (outlier superior)",
}
ETIQUETA_METRICA = ETIQUETA_METRICA_TECNICA

# ── 3c. Jerarquía temporal: padre → lista de hijos ────────────────────────
# Usada para agrupar horas bajo su franja al redactar
JERARQUIA = {
    "t_Madrugada": [f"t_H{h:02d}" for h in range(0, 7)],
    "t_Mañana":    [f"t_H{h:02d}" for h in range(7, 14)],
    "t_Tarde":     [f"t_H{h:02d}" for h in range(14, 21)],
    "t_Noche":     [f"t_H{h:02d}" for h in range(21, 24)],
}
# Mapa inverso: hora → franja
HORA_A_FRANJA = {h: f for f, hs in JERARQUIA.items() for h in hs}

# ── 3d. Categorías temporales (para agrupar reglas similares) ─────────────
HORAS   = {f"t_H{h:02d}" for h in range(24)}
FRANJAS = {"t_Madrugada", "t_Mañana", "t_Tarde", "t_Noche"}
MESES   = {"t_Ene","t_Feb","t_Marz","t_Abr","t_May","t_Jun",
            "t_Jul","t_Ago","t_Sep","t_Oct","t_Nov","t_Dic"}
DIAS    = {"t_Lun","t_Mar","t_Mie","t_Jue","t_Vie","t_Sab","t_Dom"}
ANIOS   = {"t_2024","t_2025"}
TIPO_DIA = {"t_Laborable","t_FinSemana"}
QUINCENAS = {"t_Q1mes","t_Q2mes"}
MINUTOS   = {"t_M00","t_M15","t_M30","t_M45"}
FESTIVOS  = {"t_Festivo"}
ESTACIONES = {"t_Primavera","t_Verano","t_Otonio","t_Invierno"}

print("Diccionarios cargados correctamente.")

# Validación semántica: meses válidos por estación
MESES_POR_ESTACION = {
    "t_Invierno":  {"t_Dic", "t_Ene", "t_Feb"},
    "t_Primavera": {"t_Marz", "t_Abr", "t_May"},
    "t_Verano":    {"t_Jun", "t_Jul", "t_Ago"},
    "t_Otonio":    {"t_Sep", "t_Oct", "t_Nov"},
}


# Grupos mutuamente excluyentes — dinámicos según columnas del CSV
# (robusto si src01 desactivó algún bloque)
def _construir_grupos_src03(cols_disponibles):
    cols = set(cols_disponibles)
    candidatos = [
        {"t_Ene","t_Feb","t_Marz","t_Abr","t_May","t_Jun",
         "t_Jul","t_Ago","t_Sep","t_Oct","t_Nov","t_Dic"},
        {"t_Primavera","t_Verano","t_Otonio","t_Invierno"},
        {f"t_H{h:02d}" for h in range(24)},
        {"t_Madrugada","t_Mañana","t_Tarde","t_Noche"},
        {"t_Lun","t_Mar","t_Mie","t_Jue","t_Vie","t_Sab","t_Dom"},
        {"t_Laborable","t_FinSemana"},
        {c for c in cols if c.startswith("t_20")},  # años dinámicos
        {"t_Q1mes","t_Q2mes"},
        {"t_M00","t_M15","t_M30","t_M45"},           # minutos
        {"t_Laborable","t_Sab","t_Dom"},
        {"t_FinSemana","t_Lun","t_Mar","t_Mie","t_Jue","t_Vie"},
    ]
    return [g & cols for g in candidatos if len(g & cols) >= 2]

# Se construyen tras cargar df_reglas (cols disponibles se infieren de ETIQUETA_TEMPORAL)
_cols_reglas = set(
    tok
    for ant in df_reglas["antecedente"]
    for tok in ant.split(" AND ")
)
GRUPOS_EXCLUYENTES = _construir_grupos_src03(_cols_reglas)

## 4. Funciones auxiliares de verbalización

In [ ]:
# @title
# ---------------------------------------------------------------------------
# 4. Funciones auxiliares de verbalización
# ---------------------------------------------------------------------------

def parsear_antecedente(ant_str):
    """Convierte la cadena 't_X AND t_Y' en un conjunto de tokens."""
    return set(t.strip() for t in ant_str.split(" AND "))

def es_combinacion_valida(antecedente_str):
    tokens = set(t.strip() for t in antecedente_str.split(" AND "))

    # Regla 1: dentro de cada grupo, máximo 1 valor
    for grupo in GRUPOS_EXCLUYENTES:
        if len(tokens & grupo) > 1:
            return False

    # Regla 2: mes y estación deben ser compatibles
    for estacion, meses_validos in MESES_POR_ESTACION.items():
        if estacion in tokens:
            meses_en_regla = tokens & GRUPOS_EXCLUYENTES[0]  # set de meses
            if meses_en_regla - meses_validos:
                return False

    return True


def categoria_dominante(tokens):
    """
    Devuelve la categoría temporal de mayor resolución presente en el
    antecedente, usada para agrupar reglas temáticamente.
    Orden: horas > franjas > días semana > tipo_dia > meses > estaciones >
           quincenas > años
    """
    if tokens & HORAS:      return "hora"
    if tokens & MINUTOS:    return "minuto"
    if tokens & FRANJAS:    return "franja"
    if tokens & DIAS:       return "dia_semana"
    if tokens & FESTIVOS:   return "festivo"
    if tokens & TIPO_DIA:   return "tipo_dia"
    if tokens & MESES:      return "mes"
    if tokens & ESTACIONES: return "estacion"
    if tokens & QUINCENAS:  return "quincena"
    if tokens & ANIOS:      return "anio"
    return "otro"

def dimensiones_antecedente(tokens):
    """
    Devuelve un frozenset de todas las dimensiones temporales presentes.
    Ejemplo: {t_H18, t_Lun} → frozenset({"hora", "dia_semana"})
    """
    dims = set()
    if tokens & HORAS:      dims.add("hora")
    if tokens & MINUTOS:    dims.add("minuto")
    if tokens & FRANJAS:    dims.add("franja")
    if tokens & DIAS:       dims.add("dia_semana")
    if tokens & FESTIVOS:   dims.add("festivo")
    if tokens & TIPO_DIA:   dims.add("tipo_dia")
    if tokens & MESES:      dims.add("mes")
    if tokens & ESTACIONES: dims.add("estacion")
    if tokens & QUINCENAS:  dims.add("quincena")
    if tokens & ANIOS:      dims.add("anio")
    return frozenset(dims)


def franja_de_tokens(tokens):
    """
    Si el antecedente contiene horas, devuelve la franja horaria
    a la que pertenecen (si todas coinciden).
    """
    horas_presentes = tokens & HORAS
    franjas = {HORA_A_FRANJA.get(h) for h in horas_presentes} - {None}
    return franjas.pop() if len(franjas) == 1 else None


def verbalizar_token(tok):
    """Devuelve la descripción legible de un token temporal."""
    return ETIQUETA_TEMPORAL.get(tok, tok.replace("t_", ""))


def listar_en_español(items, conector="y"):
    """
    ['a','b','c'] → 'a, b y c'
    ['a','b']     → 'a y b'
    ['a']         → 'a'
    """
    items = list(items)
    if len(items) == 0: return ""
    if len(items) == 1: return items[0]
    return ", ".join(items[:-1]) + f" {conector} " + items[-1]


def horas_consecutivas(lista_tokens_hora):
    """
    Dado ["t_H03","t_H04","t_H05"] devuelve "entre las 3 h y las 5 h".
    Si no son consecutivas devuelve lista normal.
    """
    nums = sorted(int(t[3:]) for t in lista_tokens_hora)
    if not nums:
        return ""
    # Comprobar si son consecutivas
    if nums == list(range(nums[0], nums[-1]+1)) and len(nums) > 1:
        return f"entre las {nums[0]} h y las {nums[-1]} h"
    return listar_en_español([f"las {n} h" for n in nums])


def verbalizar_antecedente(tokens):
    """
    Convierte un conjunto de tokens temporales en una frase natural.
    Ejemplos:
      {t_H03, t_H04, t_Laborable} → "entre las 3 h y las 4 h en días laborables"
      {t_Tarde, t_Lun}            → "la tarde de los lunes"
      {t_Verano}                  → "verano"
    """
    horas_tok   = sorted(tokens & HORAS)
    minutos_tok = sorted(tokens & MINUTOS)
    franjas_tok = sorted(tokens & FRANJAS)
    dias_tok    = sorted(tokens & DIAS)
    festivos_tok= sorted(tokens & FESTIVOS)
    tipo_tok    = sorted(tokens & TIPO_DIA)
    meses_tok   = sorted(tokens & MESES)
    est_tok     = sorted(tokens & ESTACIONES)
    quin_tok    = sorted(tokens & QUINCENAS)
    anio_tok    = sorted(tokens & ANIOS)

    partes = []

    # Bloque temporal principal
    # Si hay minutos Y horas: "el primer cuarto de hora de las 8h" (minuto primero)
    if minutos_tok and horas_tok:
        partes.append(listar_en_español([verbalizar_token(m) for m in minutos_tok]))
        partes.append(horas_consecutivas(horas_tok))
    elif horas_tok:
        partes.append(horas_consecutivas(horas_tok))
    elif franjas_tok:
        partes.append(listar_en_español([verbalizar_token(f) for f in franjas_tok]))
    elif minutos_tok:
        partes.append(listar_en_español([verbalizar_token(m) for m in minutos_tok]))

    # Festivos
    if festivos_tok:
        partes.append("días festivos")

    # Tipo de día / días de la semana
    if tipo_tok:
        partes.append(listar_en_español([verbalizar_token(t) for t in tipo_tok]))
    if dias_tok:
        partes.append(listar_en_español([verbalizar_token(d) for d in dias_tok]))

    # Meses
    if meses_tok:
        partes.append(listar_en_español([verbalizar_token(m) for m in meses_tok]))

    # Estaciones
    if est_tok:
        partes.append(listar_en_español([verbalizar_token(e) for e in est_tok]))

    # Quincenas
    if quin_tok:
        partes.append(listar_en_español([verbalizar_token(q) for q in quin_tok]))

    # Años
    if anio_tok:
        partes.append(listar_en_español([verbalizar_token(a) for a in anio_tok]))

    if not partes:
        return "condiciones no especificadas"

    # Unir con " en " o con ", "
    resultado = partes[0]
    for p in partes[1:]:
        resultado += " en " + p
    return resultado

# ---------------------------------------------------------------------------
# Escala adverbial — umbrales de lift fijados por el usuario
# ---------------------------------------------------------------------------
# La fuerza estadística de una regla se verbaliza con cuatro adverbios.


def construir_calidad(df_reglas=None):
    """
    Devuelve una función calidad(row) que traduce el lift de una regla en
    un adverbio de fuerza estadística."""

    def calidad(row):
        lift = row.get("lift", 0)

        if lift >= 3.0:
            return "de forma muy marcada"
        elif lift >= 2.0:
            return "de forma notable"
        elif lift >= 1.5:
            return "con cierta consistencia"
        else:
            return "con cierta tendencia"
    return calidad


## 5. Generación de frases individuales y párrafos agrupados

In [ ]:
# ---------------------------------------------------------------------------
# 5a. Generación de una frase para una sola regla
# ---------------------------------------------------------------------------

def regla_a_frase(row, nombre_metrica, modo="tecnico"):
    """
    Convierte una fila del DataFrame de reglas en una frase en español.

    modo="tecnico":  incluye confianza y lift entre paréntesis (apéndice).
    modo="coloquial": omite las cifras (cuerpo del informe).
    """
    tokens  = parsear_antecedente(row["antecedente"])
    cat     = categoria_dominante(tokens)
    desc_t  = verbalizar_antecedente(tokens)

    diccionario = (ETIQUETA_METRICA_COLOQUIAL if modo == "coloquial"
                   else ETIQUETA_METRICA_TECNICA)
    desc_v  = diccionario.get(row["consecuente"], row["consecuente"])
    calidad = construir_calidad(df_reglas)(row)

    prefijos = {
        "hora":       "A",
        "franja":     "Durante",
        "dia_semana": "En",
        "tipo_dia":   "Durante los",
        "mes":        "En",
        "estacion":   "En",
        "quincena":   "Durante",
        "anio":       "En",
        "otro":       "En",
    }
    prefijo = prefijos.get(cat, "En")

    if modo == "tecnico":
        conf_pct = int(round(row["confianza"] * 100))
        lift_val = f"{row['lift']:.1f}"
        return (f"{prefijo} {desc_t}, la {nombre_metrica} tiende a ser {desc_v} "
                f"{calidad} (confianza {conf_pct} %, lift {lift_val}).")
    else:
        return (f"{prefijo} {desc_t}, la {nombre_metrica} tiende a ser {desc_v} "
                f"{calidad}.")


# ---------------------------------------------------------------------------
# 5b. Agrupación de reglas relacionadas en un párrafo
# ---------------------------------------------------------------------------

def agrupar_reglas(df_consecuente, umbral_solapamiento=0.4):
    """
    Agrupa reglas con el mismo consecuente y contexto similar en un párrafo.

    FIX v3: umbral de solapamiento bajado de 0.5 → 0.4 para recuperar el
    agrupamiento que se perdía al ordenar por lift antes de iterar.
    El primer registro (mayor lift) actúa como ancla del grupo; el resto
    se adhiere si comparte al menos el 40 % de tokens (excl. años/festivos).

    Parámetro umbral_solapamiento: puede sobreescribirse por caller
    (p.ej. 0.5 si se quiere agrupamiento más estricto en tests futuros).
    """
    # El DataFrame ya viene ordenado por lift DESC desde generar_resumen
    registros = list(df_consecuente.iterrows())
    grupos = []
    asignado = [False] * len(registros)

    for i, (_, fila_i) in enumerate(registros):
        if asignado[i]:
            continue
        tokens_i = parsear_antecedente(fila_i["antecedente"])
        cat_i    = categoria_dominante(tokens_i)

        if cat_i == "hora":
            franja_i = franja_de_tokens(tokens_i)
        else:
            franja_i = None

        grupo_actual = [fila_i]
        asignado[i]  = True

        for j, (_, fila_j) in enumerate(registros):
            if asignado[j]:
                continue
            tokens_j = parsear_antecedente(fila_j["antecedente"])
            cat_j    = categoria_dominante(tokens_j)

            if cat_i != cat_j:
                continue
            if cat_i == "hora":
                franja_j = franja_de_tokens(tokens_j)
                if franja_i != franja_j:
                    continue
            if cat_i == "minuto":
                if (tokens_i & HORAS) != (tokens_j & HORAS):
                    continue
            if cat_i == "franja":
                if (tokens_i & FRANJAS) != (tokens_j & FRANJAS):
                    continue
            if cat_i == "dia_semana":
                if (tokens_i & DIAS) != (tokens_j & DIAS):
                    continue

            _excluir = ANIOS | FESTIVOS
            ti = tokens_i - _excluir if (tokens_i - _excluir) else tokens_i
            tj = tokens_j - _excluir if (tokens_j - _excluir) else tokens_j
            comun   = ti & tj
            solap_i = len(comun) / len(ti) if ti else 0
            solap_j = len(comun) / len(tj) if tj else 0
            if min(solap_i, solap_j) < umbral_solapamiento:
                continue

            grupo_actual.append(fila_j)
            asignado[j] = True

        grupos.append(grupo_actual)

    return grupos

def grupo_a_parrafo(filas, nombre_metrica, consecuente, min_reglas, modo="tecnico"):
    """
    Convierte un grupo de filas (mismo consecuente + contexto similar) en un
    párrafo narrativo. Si el grupo tiene < min_reglas, genera frases sueltas.

    modo="tecnico":   incluye confianza y lift entre paréntesis (apéndice).
    modo="coloquial": omite las cifras técnicas (cuerpo del informe).
    """
    diccionario = (ETIQUETA_METRICA_COLOQUIAL if modo == "coloquial"
                   else ETIQUETA_METRICA_TECNICA)
    desc_v = diccionario.get(consecuente, consecuente)
    filas_ordenadas = sorted(filas, key=lambda r: -r["lift"])

    # ── Caso A: grupo pequeño → frases individuales ───────────────────────
    if len(filas_ordenadas) < min_reglas:
        return "\n".join(regla_a_frase(f, nombre_metrica, modo=modo)
                         for f in filas_ordenadas)

    # ── Caso B: grupo amplio → párrafo narrativo ──────────────────────────
    conjuntos = [parsear_antecedente(f["antecedente"]) for f in filas_ordenadas]
    contexto_comun = conjuntos[0].copy()
    for c in conjuntos[1:]:
        contexto_comun &= c

    # Calcular tokens específicos de cada regla
    especificos = [c - contexto_comun for c in conjuntos]

    # ── Caso B.1: los detalles diferenciales son solo años → frase simple ──
    solo_anios = all(e <= ANIOS or not e for e in especificos)
    if solo_anios:
        desc_ctx = verbalizar_antecedente(contexto_comun) if contexto_comun \
                   else verbalizar_antecedente(conjuntos[0])
        prefijo  = "A" if contexto_comun & HORAS else "En"

        if modo == "tecnico":
            conf_media = sum(f["confianza"] for f in filas_ordenadas) / len(filas_ordenadas)
            lift_max   = max(f["lift"] for f in filas_ordenadas)
            return (
                f"{prefijo} {desc_ctx}, la {nombre_metrica} tiende a ser {desc_v} "
                f"(confianza media {int(round(conf_media*100))} %, "
                f"lift máximo {lift_max:.1f})."
            )
        else:  # coloquial
            return (
                f"{prefijo} {desc_ctx}, la {nombre_metrica} tiende a ser {desc_v}."
            )

    # ── Caso B.2: párrafo narrativo completo ──────────────────────────────
    detalles = []
    for f, especif in zip(filas_ordenadas, especificos):
        if especif:
            especif_filtrado = especif - ANIOS if (especif - ANIOS) else especif
            desc = verbalizar_antecedente(especif_filtrado)
        else:
            desc = verbalizar_antecedente(parsear_antecedente(f["antecedente"]))

        if modo == "tecnico":
            conf_pct = int(round(f["confianza"] * 100))
            detalles.append(f"{desc} ({conf_pct} %)")
        else:  # coloquial
            detalles.append(desc)

    if contexto_comun:
        ctx_desc = verbalizar_antecedente(contexto_comun)
        intro = (
            f"En el contexto de {ctx_desc}, la {nombre_metrica} tiende a ser "
            f"{desc_v}, especialmente en {listar_en_español(detalles)}."
        )
    else:
        intro = (
            f"La {nombre_metrica} tiende a ser {desc_v} en los siguientes "
            f"contextos: {listar_en_español(detalles)}."
        )

    if modo == "tecnico":
        conf_media = sum(f["confianza"] for f in filas_ordenadas) / len(filas_ordenadas)
        lift_max   = max(f["lift"] for f in filas_ordenadas)
        stats = (
            f"Este patrón se observa con una confianza media del "
            f"{int(round(conf_media*100))} % y un lift máximo de {lift_max:.1f}."
        )
        return intro + " " + stats
    else:  # coloquial
        return intro

print("Funciones de generación de párrafos definidas.")



## 6. Pipeline principal: reglas → resumen completo

In [ ]:
# ---------------------------------------------------------------------------
# 6a. Narrativa diaria — sección accesible para cualquier lector
# ---------------------------------------------------------------------------
# Genera DOS secciones nuevas que se añaden AL INICIO del resumen:
#
#   1. "¿Cómo se comporta la metrica a lo largo del día?" (lenguaje llano,
#      sin cifras técnicas, apta para cualquier usuario del sistema)
#   2. "Análisis por franja horaria" (técnico pero organizado por momento
#      del día, en lugar de por nivel de metrica)
#
# Luego continúa con la sección técnica existente como apéndice de referencia.
# ---------------------------------------------------------------------------

# ── Constantes compartidas ──────────────────────────────────────────────────
HORAS_FRANJA_MAP = {
    "t_Madrugada": list(range(0, 7)),
    "t_Mañana":    list(range(7, 14)),
    "t_Tarde":     list(range(14, 21)),
    "t_Noche":     list(range(21, 24)),
}
NOMBRE_FRANJA = {
    "t_Madrugada": "Madrugada (0–6 h)",
    "t_Mañana":    "Mañana (7–13 h)",
    "t_Tarde":     "Tarde (14–20 h)",
    "t_Noche":     "Noche (21–23 h)",
}
NIVEL_COLOQUIAL = {
    "v_OutlierAlto": "excepcionalmente alto (un pico)",
    "v_MuyAlta": "muy alto",
    "v_Alta":    "alto",
    "v_Media":   "moderado",
    "v_Baja":    "bajo",
    "v_MuyBaja": "muy bajo",
    "v_OutlierBajo": "excepcionalmente bajo (un valle)",
}
NIVEL_EMOJI = {
    "v_OutlierAlto": "🔴",
    "v_MuyAlta":     "🟠",
    "v_Alta":        "🟡",
    "v_Media":       "🟢",
    "v_Baja":        "🔵",
    "v_MuyBaja":     "🟣",
    "v_OutlierBajo": "⚪",
}
# Peso de cada nivel para elegir el representativo de cada franja
# (mayor número = más relevante para el lector)
NIVEL_PESO = {
    "v_OutlierAlto": 7, "v_MuyAlta": 6, "v_Alta": 5,
    "v_Media": 4,
    "v_Baja": 3, "v_MuyBaja": 2, "v_OutlierBajo": 1,
}

# ── Función auxiliar: consecuente dominante para una hora concreta ──────────
def _consecuente_hora(df_reglas, hora):
    """Devuelve (consecuente, lift) con más lift para una hora específica."""
    tok = f"t_H{hora:02d}"
    mask = df_reglas["antecedente"].str.contains(tok, regex=False)
    sub = df_reglas[mask]
    if sub.empty:
        return None
    idx = sub["lift"].idxmax()
    return sub.loc[idx, "consecuente"], sub.loc[idx, "lift"]


def _consecuente_franja(df_reglas, franja_tok):
    """Devuelve (consecuente, lift) más fuerte para una franja entera."""
    mask = df_reglas["antecedente"].str.contains(franja_tok, regex=False)
    sub = df_reglas[mask]
    if sub.empty:
        return None
    idx = sub["lift"].idxmax()
    return sub.loc[idx, "consecuente"], sub.loc[idx, "lift"]


# ── Función: construir mapa hora → nivel dominante (añadida) ───────────────
def _mapa_horario(df_reglas):
    """
    Construye un diccionario {hora: consecuente_dominante} basado en la regla
    con mayor lift para cada hora.
    """
    mapa = {}
    for h in range(24):
        tok = f"t_H{h:02d}"
        # Filtra reglas que contengan el token de la hora específica
        mask = df_reglas["antecedente"].str.contains(tok, regex=False)
        sub = df_reglas[mask]
        if not sub.empty:
            # Encuentra el consecuente con el lift más alto para esa hora
            idx = sub["lift"].idxmax()
            mapa[h] = sub.loc[idx, "consecuente"]
    return mapa

def _nota_fds(df_reglas, mapa_horario):
    """
    Genera la nota de fin de semana solo si hay reglas que la justifiquen,
    y distingue entre 'inversión' y 'reducción'.
    """
    NIVELES_ALTOS = {"v_OutlierAlto", "v_MuyAlta", "v_Alta"}
    NIVELES_BAJOS = {"v_Baja", "v_MuyBaja", "v_OutlierBajo"}

    hay_fds = any(
        "t_FinSemana" in a or "t_Sab" in a or "t_Dom" in a
        for a in df_reglas["antecedente"]
    )
    if not hay_fds:
        return None

    # Detectar si alguna franja laboral alta tiene contraparte baja en fds
    hay_inversion = False
    for franja_tok, horas in HORAS_FRANJA_MAP.items():
        niveles_lab = [
            mapa_horario[h] for h in horas
            if h in mapa_horario
            and any("t_Laborable" in a and f"t_H{h:02d}" in a
                    for a in df_reglas["antecedente"])
        ]
        niveles_fds = [
            mapa_horario[h] for h in horas
            if h in mapa_horario
            and any(
                ("t_FinSemana" in a or "t_Sab" in a or "t_Dom" in a)
                and f"t_H{h:02d}" in a
                for a in df_reglas["antecedente"]
            )
        ]
        if any(n in NIVELES_ALTOS for n in niveles_lab) and \
           any(n in NIVELES_BAJOS for n in niveles_fds):
            hay_inversion = True
            break

    if hay_inversion:
        return (
            "Los **fines de semana** invierten el patrón laboral: "
            "las franjas de mayor actividad entre semana presentan "
             +METRICA+" sensiblemente más bajo."
        )
    else:
        return (
            "Los **fines de semana** el "+METRICA+" se reduce respecto a los días laborables, "
            "aunque sin llegar a invertir el patrón general."
        )


def _parrafo_coloquial(df_reglas, mapa_horario, nombre_metrica, nombre_franja=NOMBRE_FRANJA):
    """
    Genera prosa narrativa por franjas, variando la estructura sintáctica
    para evitar repetición y fusionando el contexto de forma fluida.
    """

    def _nivel_franja(horas):
      # Recoge TODAS las reglas de la franja, no solo el pico por hora
      patron = "|".join(f"t_H{h:02d}" for h in horas)
      sub = df_reglas[df_reglas["antecedente"].str.contains(patron, regex=True)]
      if sub.empty:
          return None

      # Suma lift × confianza por nivel
      votos: dict[str, float] = {}
      for _, row in sub.iterrows():
          n = row["consecuente"]
          votos[n] = votos.get(n, 0.0) + row["lift"] * row["confianza"]

      return max(votos, key=votos.get)

    def _ctx_franja(sub, horas):
        """Devuelve dict con flags de contexto para la franja."""
        ants = list(sub["antecedente"])
        return {
            "lab": any("t_Laborable" in a for a in ants),
            "fds": any(("t_FinSemana" in a or "t_Sab" in a or "t_Dom" in a) for a in ants),
            "ago": any("t_Ago" in a for a in ants),
            "fes": any("t_Festivo" in a for a in ants),
        }

    def _outlier_matiz(nivel_clave, horas):
        """Devuelve texto del matiz puntual si aplica, o ''."""
        niveles_en_franja = [mapa_horario.get(h) for h in horas if h in mapa_horario]
        if "v_OutlierAlto" in niveles_en_franja and nivel_clave != "v_OutlierAlto":
            h = next(h for h in horas if mapa_horario.get(h) == "v_OutlierAlto")
            return f"pico excepcional a las {h} h"
        if "v_OutlierBajo" in niveles_en_franja and nivel_clave != "v_OutlierBajo":
            h = next(h for h in horas if mapa_horario.get(h) == "v_OutlierBajo")
            return f"mínimo absoluto a las {h} h"
        return ""

    # Plantillas con estructura sintáctica diferente por franja
    # {rango}, {nivel}, {emoji}, {matiz}, {ctx_*}
    def _frase_madrugada(rango, nivel, emoji, matiz, ctx, **kwargs):
        base = f"De madrugada ({rango}), el "+METRICA+" es **{nivel}** {emoji}"
        detalles = []
        if matiz:
            if ctx["lab"]:
                detalles.append(f"llegando a ser {matiz} en laborables")
            else:
                detalles.append(matiz)
        elif ctx["lab"] and ctx["fds"]:
            detalles.append("más pronunciado en días laborables que en fin de semana")
        if ctx["ago"]:
            detalles.append("con variación en agosto")
        sufijo = (", " + ", ".join(detalles)) if detalles else ""
        return base + sufijo + "."

    def _frase_manana(rango, nivel, emoji, matiz, ctx, nivel_prev=None, nivel_clave=None):
      # Verbo según comparación con franja previa (madrugada)
      if nivel_prev is not None and nivel_clave is not None:
          p_act  = NIVEL_PESO.get(nivel_clave, 0)
          p_prev = NIVEL_PESO.get(nivel_prev, 0)
          if p_act > p_prev + 1:
              # Si venimos de un outlier bajo, matizamos la "subida"
              if nivel_prev == "v_OutlierBajo":
                  verbo = "mejora hasta niveles"
              else:
                  verbo = "sube hasta niveles"
          elif p_act < p_prev - 1:
              verbo = "baja a niveles"
          elif p_act == p_prev:
              verbo = "se mantiene en niveles"
          else:
              verbo = "se sitúa en niveles"
      else:
          verbo = "alcanza niveles"

      base = f"Al llegar la mañana ({rango}), el {METRICA} {verbo} **{nivel}** {emoji}"
      detalles = []
      if ctx["lab"] and ctx["fds"]:
          detalles.append("en días laborables; se reduce sensiblemente en fin de semana")
      elif ctx["lab"]:
          detalles.append("sobre todo en días laborables")
      if ctx["fes"]:
          detalles.append("y cae en festivos")
      if ctx["ago"]:
          detalles.append("con un patrón distinto en agosto")
      sufijo = (", " + ", ".join(detalles)) if detalles else ""
      return base + sufijo + "."


    def _frase_tarde(rango, nivel, emoji, matiz, ctx, nivel_prev=None, nivel_clave=None):
        if nivel_prev is not None and nivel_clave is not None:
            p_act = NIVEL_PESO.get(nivel_clave, 0)
            p_prev = NIVEL_PESO.get(nivel_prev, 0)
            if p_act > p_prev + 1:
                verbo = "sube a niveles"
            elif p_act < p_prev - 1:
                verbo = "desciende a niveles"
            elif p_act == p_prev:
                verbo = "se mantiene"
            else:
                verbo = "queda en niveles"
        else:
            verbo = "se sitúa en niveles"

        base = f"Por la tarde ({rango}), el {METRICA} {verbo} **{nivel}** {emoji}"
        detalles = []
        if ctx["lab"] and ctx["fds"]:
            detalles.append("especialmente entre semana, con calma en fin de semana")
        elif ctx["lab"]:
            detalles.append("con mayor intensidad en días laborables")
        if ctx["ago"]:
            detalles.append("aunque agosto rompe este patrón")
        sufijo = (", " + ", ".join(detalles)) if detalles else ""
        return base + sufijo + "."


    def _frase_noche(rango, nivel, emoji, matiz, ctx, nivel_prev=None, nivel_clave=None):
        if nivel_prev is not None and nivel_clave is not None:
            p_act = NIVEL_PESO.get(nivel_clave, 0)
            p_prev = NIVEL_PESO.get(nivel_prev, 0)
            if p_act < p_prev - 1:
                verbo = "cae a un nivel"
            elif p_act > p_prev + 1:
                verbo = "repunta a un nivel"
            elif p_act == p_prev:
                verbo = "se mantiene en un nivel"
            else:
                verbo = "queda en un nivel"
        else:
            verbo = "se sitúa en un nivel"

        base = f"Al caer la noche ({rango}), el {METRICA} {verbo} **{nivel}** {emoji}"
        detalles = []
        if matiz:
            detalles.append(f"con un {matiz}")
        if ctx["fds"]:
            detalles.append("algo más escaso en fin de semana")
        sufijo = (", " + ", ".join(detalles)) if detalles else ""
        return base + sufijo + "."

    PLANTILLAS = {
        "t_Madrugada": _frase_madrugada,
        "t_Mañana":    _frase_manana,
        "t_Tarde":     _frase_tarde,
        "t_Noche":     _frase_noche,
    }

    frases = []
    nivel_prev = None    # ← guardar el nivel de la franja anterior
    for franja_tok, horas in HORAS_FRANJA_MAP.items():
        nivel_clave = _nivel_franja(horas)
        if nivel_clave is None:
            nombre_corto = NOMBRE_FRANJA[franja_tok].split(" ")[0].lower()
            frases.append(
                f"{'Por la' if nombre_corto in ('tarde','noche') else 'Durante la'} "
                f"{nombre_corto} el sistema no detecta ningún patrón temporal estable."
            )
            nivel_prev = None    # ← resetear si hay hueco
            continue

        nivel = NIVEL_COLOQUIAL[nivel_clave]
        emoji = NIVEL_EMOJI[nivel_clave]
        rango = f"{horas[0]}–{horas[-1]} h"

        patron_horas = "|".join([f"t_H{h:02d}" for h in horas])
        mask = (
            df_reglas["antecedente"].str.contains(franja_tok, regex=False) |
            df_reglas["antecedente"].str.contains(patron_horas, regex=True)
        )
        ctx   = _ctx_franja(df_reglas[mask], horas)
        matiz = _outlier_matiz(nivel_clave, horas)

        frase = PLANTILLAS[franja_tok](
            rango, nivel, emoji, matiz, ctx,
            nivel_prev=nivel_prev, nivel_clave=nivel_clave
        )
        frases.append(frase)
        nivel_prev = nivel_clave    # ← guardar para la próxima iteración

    texto = " ".join(frases)

    notas = []

    # Llamada a la nueva función lógica para Fines de Semana
    nota_fds = _nota_fds(df_reglas, mapa_horario)
    if nota_fds:
        notas.append(nota_fds)

    # Mantener la nota de festivos si existe el token
    tiene_fes = any("t_Festivo" in a for a in df_reglas["antecedente"])
    if tiene_fes:
        notas.append(
            "Los **festivos** se comportan de forma similar al fin de semana "
            "con independencia del día de la semana en que caigan."
        )

    if notas:
        texto += "\n\n" + " ".join(notas)

    return texto




# ── Función: detalle técnico por franja ────────────────────────────────────
def _detalle_por_franja(df_reglas, nombre_metrica, min_reglas_grupo=2):
    """
    Sección técnica organizada por franja horaria.
    Dentro de cada franja, reglas ordenadas por consecuente (mayor→menor) y lift.
    """
    lineas = []
    ORDEN = ["v_OutlierAlto","v_MuyAlta","v_Alta","v_Media",
             "v_Baja","v_MuyBaja","v_OutlierBajo"]

    for franja_tok, horas in HORAS_FRANJA_MAP.items():
        nombre_f = NOMBRE_FRANJA[franja_tok]
        patron_horas = "|".join([f"t_H{h:02d}" for h in horas])
        mask = (
            df_reglas["antecedente"].str.contains(franja_tok, regex=False) |
            df_reglas["antecedente"].str.contains(patron_horas, regex=True)
        )
        sub = df_reglas[mask].copy()
        if sub.empty:
            continue

        sub = sub.sort_values("lift", ascending=False)
        lineas.append(f"### {nombre_f}")
        lineas.append("")

        for cons in ORDEN:
            sub_c = sub[sub["consecuente"] == cons]
            if sub_c.empty:
                continue
            desc_v = ETIQUETA_METRICA.get(cons, cons)
            emoji  = NIVEL_EMOJI.get(cons, "")
            for _, row in sub_c.iterrows():
                tokens   = parsear_antecedente(row["antecedente"])
                desc_t   = verbalizar_antecedente(tokens)
                conf_pct = int(round(row["confianza"] * 100))
                lift_val = f"{row['lift']:.1f}"
                cal      = construir_calidad(df_reglas)(row)
                lineas.append(
                    f"- {emoji} **{desc_v.capitalize()}** {cal}: "
                    f"{desc_t} (confianza {conf_pct}\u202f%, lift {lift_val})"
                )
        lineas.append("")


    return "\n".join(lineas)

def _parrafo_calendario(df_reglas, nombre_metrica, lift_min=3.0, top_n=3):
    """
    Genera un párrafo con las reglas no horarias más fuertes (mes, estación, año).
    Solo se incluye cuando existen reglas con lift >= lift_min.
    """
    TOKENS_NO_HORARIOS = MESES | ESTACIONES | ANIOS
    TOKENS_HORARIOS    = HORAS | FRANJAS

    mask = df_reglas["antecedente"].apply(
        lambda a: bool(parsear_antecedente(a) & TOKENS_NO_HORARIOS)
                  and not bool(parsear_antecedente(a) & TOKENS_HORARIOS)
    )
    sub = df_reglas[mask & (df_reglas["lift"] >= lift_min)] \
            .sort_values("lift", ascending=False) \
            .head(top_n)

    if sub.empty:
        return None

    lineas = ["## Patrones estacionales y de calendario", ""]
    lineas.append(
        "_Los siguientes patrones no están ligados a una franja horaria concreta "
        "sino a períodos del calendario con comportamiento diferencial._"
    )
    lineas.append("")
    for _, row in sub.iterrows():
        tokens  = parsear_antecedente(row["antecedente"])
        desc_t  = verbalizar_antecedente(tokens)
        desc_v  = ETIQUETA_METRICA.get(row["consecuente"], row["consecuente"])
        emoji   = NIVEL_EMOJI.get(row["consecuente"], "")
        conf    = int(round(row["confianza"] * 100))
        lineas.append(
            f"- {emoji} En **{desc_t}**, la {nombre_metrica} tiende a ser "
            f"**{desc_v}** (confianza {conf} %, lift {row['lift']:.1f})."
        )
    return "\n".join(lineas)

def _generar_glosario_breve():
    """Glosario corto, sin jerga técnica — para la cabecera."""
    return (
        "> ### 📖 Niveles de la métrica analizada ("+METRICA+") \n"
        "> Escala desde **excepcionalmente bajo** ⚪ hasta **excepcionalmente alto** 🔴, "
        "calculada a partir de la desviación sobre la media histórica del propio sensor.\n"
        ">\n"
        "> Cuando el patrón es *muy diferenciado* del comportamiento normal se describe "
        "*'de forma notable'* o *'muy marcada'*. Cuando es solo una *tendencia*, se dice así."
    )

def _generar_glosario_tecnico():
    """Glosario completo — para insertar al INICIO del apéndice técnico."""
    return (
        "> ### 📖 Glosario técnico (apéndice)\n"
        "> * **Confianza:** probabilidad (0–100 %) de que el patrón sea cierto cuando "
        "se da el contexto. Confianza 80 % significa que, en ese contexto, el la métrica "
        "se comporta así 8 de cada 10 veces.\n"
        "> * **Lift:** cuántas veces más probable es el evento respecto al azar. "
        "Lift 5.0 significa que el patrón es 5 veces más frecuente en ese contexto "
        "que en el resto del tiempo.\n"
        "> * **Outlier inferior/superior:** valor más allá de 2 desviaciones típicas "
        "de la media histórica del sensor."
    )


print("Funciones de narrativa definidas:")
print("  · _parrafo_coloquial()  — 4 bullets compactos (uno por franja)")
print("  · _detalle_por_franja() — detalle técnico por momento del día")
print("  · _mapa_horario()       — crea el mapa de horas a niveles dominantes")

In [ ]:
# ---------------------------------------------------------------------------
# 6b. generar_resumen — narrativa + técnico + apéndice con agrupamiento
# ---------------------------------------------------------------------------

def generar_resumen(df_reglas, dataset, metrica, min_reglas_grupo=2):
    """
    Genera el resumen completo en Markdown.

    Estructura:
      1. Cabecera (Limpia)
      2. Narrativa Coloquial (Lo más importante)
         - Aquí llamas a _parrafo_coloquial
         - Aquí llamas a _parrafo_calendario
      3. Nota de Fines de Semana (Condicional)
      4. Apéndice Técnico (Solo para quien quiera ver los porcentajes y el lift)
    """
    nombre_metrica = metrica.replace("_", " ")


    # Filtrar combinaciones semánticamente inválidas
    n_antes = len(df_reglas)
    df_reglas = df_reglas[
        df_reglas["antecedente"].apply(es_combinacion_valida)
    ].reset_index(drop=True)
    n_filtradas = n_antes - len(df_reglas)
    if n_filtradas > 0:
        print(f"☢️  {n_filtradas} regla(s) eliminadas por combinación inválida.")

    # Ordenar por lift DESC — necesario para que agrupar_reglas sea determinista
    df_reglas = df_reglas.sort_values("lift", ascending=False).reset_index(drop=True)

    mapa_horario = _mapa_horario(df_reglas)

    lineas = []

    # ── 1. Cabecera ─────────────────────────────────────────────────────────
    lineas.append(f"# Resumen de comportamiento — {NOMBRE_DATASET}")
    lineas.append("")
    lineas.append(f"**Métrica analizada:** {nombre_metrica.capitalize()}  ")
    lineas.append(f"**Total de reglas analizadas:** {len(df_reglas)}")
    lineas.append("")

    lineas.append(_generar_glosario_breve())

    # ── Visualizaciones ─────────────────────────────────────────────────────
    lineas.append("## Visualizaciones")
    lineas.append("")
    lineas.append("### Mapa de calor hora × día de la semana")
    lineas.append(f"![Mapa de calor hora x día](data/{NOMBRE_DATASET}_heatmap.png)")
    lineas.append("")
    lineas.append("### Reglas por fuerza de asociación (lift)")
    lineas.append(f"![Barras lift](data/{NOMBRE_DATASET}_barras_lift.png)")
    lineas.append("")
    lineas.append("### Soporte vs Confianza")
    lineas.append(f"![Scatter soporte-confianza](data/{NOMBRE_DATASET}_scatter.png)")
    lineas.append("")
    lineas.append("### Resumen por categoría")
    lineas.append(f"![Tabla consecuentes](data/{NOMBRE_DATASET}_tabla_consecuentes.png)")
    lineas.append("")

    # ── 2. Narrativa coloquial ──────────────────────────────────────────────
    if HAY_HORAS or HAY_FRANJAS:
        lineas.append(f"## ¿Cómo se comporta **{nombre_metrica}** a lo largo del día?")
        lineas.append(f"> Esta sección resume el comportamiento de **{nombre_metrica}** "
                      f"en lenguaje sencillo, sin tecnicismos.")
        lineas.append("")
        lineas.append(_parrafo_coloquial(df_reglas, mapa_horario, nombre_metrica))
    else:
        lineas.append(f"## Patrones detectados")
        lineas.append(
            f"> ℹ️ Este dataset tiene granularidad **{_granularidad_desc}** — "
            f"no hay patrones por franja horaria. "
            f"Los patrones detectados son {'estacionales y de calendario' if HAY_MESES or HAY_ESTACIONES else 'temporales'}."
        )
    lineas.append("")

    parrafo_cal = _parrafo_calendario(df_reglas, nombre_metrica)
    if parrafo_cal:
        lineas.append(parrafo_cal)
        lineas.append("")


    # ── 3. Análisis técnico por franja ──────────────────────────────────────
    if HAY_HORAS or HAY_FRANJAS:
        lineas.append("## Análisis por franja horaria")
        lineas.append("")
        lineas.append(
            "Detalle de los patrones detectados, organizados por momento del día. "
            "Cada punto indica el nivel de "+METRICA+" más probable en ese contexto, "
            "junto con la confianza y el lift de la regla."
        )
        lineas.append("")
        lineas.append(_detalle_por_franja(df_reglas, nombre_metrica, min_reglas_grupo))

    # ── 4. Apéndice ─────────────────────────────────────────────────────────
    lineas.append("---")
    lineas.append("")
    lineas.append("## Apéndice — Análisis por nivel de "+METRICA)
    lineas.append("")
    lineas.append("")
    lineas.append(_generar_glosario_tecnico())
    lineas.append("")
    lineas.append(
        f"*Mismas reglas organizadas por nivel de **{METRICA}** (de mayor a menor valor).*"
    )
    lineas.append("")

    ORDEN_CONSECUENTE = [
        "v_OutlierAlto", "v_MuyAlta", "v_Alta",
        "v_Media",
        "v_Baja", "v_MuyBaja", "v_OutlierBajo",
    ]
    consecuentes_en_datos = df_reglas["consecuente"].unique()
    consecuentes_ordenados = [c for c in ORDEN_CONSECUENTE if c in consecuentes_en_datos]
    for c in consecuentes_en_datos:
        if c not in consecuentes_ordenados:
            consecuentes_ordenados.append(c)

    for consecuente in consecuentes_ordenados:
        df_c = df_reglas[df_reglas["consecuente"] == consecuente].copy()
        if df_c.empty:
            continue

        desc_v = ETIQUETA_METRICA.get(consecuente, consecuente)
        emoji  = NIVEL_EMOJI.get(consecuente, "")
        # FIX: umbral de solapamiento 0.4 para recuperar agrupamiento
        # tras el sort por lift. El sort es necesario para que el párrafo
        # narrativo use la regla más fuerte como ancla del grupo.
        grupos   = agrupar_reglas(df_c, umbral_solapamiento=0.4)
        n_grupos = len(grupos)

        lineas.append(f"### {emoji} {nombre_metrica.capitalize()} {desc_v}")
        lineas.append(
            f"*{len(df_c)} {'regla' if len(df_c)==1 else 'reglas'} — "
            f"agrupadas en {n_grupos} {'contexto' if n_grupos==1 else 'contextos'}* | "
            f"confianza media: {df_c['confianza'].mean()*100:.0f} %, "
            f"lift medio: {df_c['lift'].mean():.1f} "

        )
        lineas.append("")

        for grupo in grupos:
            parrafo = grupo_a_parrafo(grupo, nombre_metrica, consecuente, min_reglas_grupo)
            lineas.append(parrafo)
            lineas.append("")


    # ── 5. Estadísticas globales ─────────────────────────────────────────────
    lineas.append("---")
    lineas.append("")
    lineas.append("## Estadísticas globales del análisis")
    lineas.append("")
    lineas.append("| Métrica | Valor |")
    lineas.append("|---|---|")
    lineas.append(f"| Reglas totales | {len(df_reglas)} |")
    lineas.append(f"| Consecuentes únicos | {len(consecuentes_ordenados)} |")
    lineas.append(f"| Soporte medio | {df_reglas['soporte'].mean():.4f} |")
    lineas.append(f"| Confianza media | {df_reglas['confianza'].mean()*100:.1f} % |")
    lineas.append(f"| Lift medio | {df_reglas['lift'].mean():.2f} |")
    lineas.append(f"| Lift máximo | {df_reglas['lift'].max():.2f} |")

    return "\n".join(lineas)


print("generar_resumen v3 — con párrafo coloquial compacto y agrupamiento recuperado.")


In [ ]:
# ---------------------------------------------------------------------------
# NUEVA CELDA — Visualizaciones de reglas para src03
# Pegar después de la celda de definición de generar_resumen (celda 11)
# ---------------------------------------------------------------------------
# Requiere en imports (celda 2): matplotlib.pyplot as plt, numpy as np
# ---------------------------------------------------------------------------

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np


# Paleta coherente con los colores del heatmap
COLORES_CONSECUENTE = {
    "v_OutlierBajo": "#4a0e8f",
    "v_MuyBaja":     "#2166ac",
    "v_Baja":        "#74add1",
    "v_Media":       "#ffffbf",
    "v_Alta":        "#fdae61",
    "v_MuyAlta":     "#d73027",
    "v_OutlierAlto": "#7f0000",
}

ETIQUETA_CONSECUENTE = {
    "v_OutlierBajo": "Outlier bajo",
    "v_MuyBaja":     "Muy baja",
    "v_Baja":        "Baja",
    "v_Media":       "Media",
    "v_Alta":        "Alta",
    "v_MuyAlta":     "Muy alta",
    "v_OutlierAlto": "Outlier alto",
}


DIAS_SEMANA = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]


# ---------------------------------------------------------------------------
# 2. Heatmap
# ---------------------------------------------------------------------------
def generar_heatmap(sensor, metrica, guardar=True):
    """
    Genera el mapa de calor hora × día de la semana para un sensor y métrica.
    Muestra la categoría difusa dominante en cada franja horaria.
    """
    fichero_fuzzy = f"data/{NOMBRE_DATASET}_{METRICA}_fuzzy.csv"
    df = pd.read_csv(fichero_fuzzy)

    # ── Identificar columnas de categoría disponibles ────────────────────
    # Orden semántico de menor a mayor (para resolver empates: gana la más alta)
    orden = ["v_OutlierBajo", "v_MuyBaja", "v_Baja",
             "v_Media", "v_Alta", "v_MuyAlta", "v_OutlierAlto"]
    cols_v = [c for c in orden if c in df.columns
              and not c.startswith("v_abs_")
              and c != "v_Mediana"]

    # ── Extraer hora y día de semana desde las columnas fuzzy ────────────
    # Las columnas t_H00..t_H23 y t_Lun..t_Dom son binarias (0/1 en src01)
    # Reconstruimos hora y día a partir de cuál columna tiene valor 1
    horas_cols = [f"t_H{h:02d}" for h in range(24) if f"t_H{h:02d}" in df.columns]
    dias_cols  = ["t_Lun","t_Mar","t_Mie","t_Jue","t_Vie","t_Sab","t_Dom"]
    dias_cols  = [c for c in dias_cols if c in df.columns]

    # --- ADDED CHECK FOR MISSING COLUMNS ---
    if not horas_cols:
        print(f"Advertencia: No se encontraron columnas de horas (t_Hxx) en el archivo fuzzy '{fichero_fuzzy}'. No se generará el heatmap horario.")
        return None
    if not dias_cols:
        print(f"Advertencia: No se encontraron columnas de días de la semana (t_Lun..t_Dom) en el archivo fuzzy '{fichero_fuzzy}'. No se generará el heatmap diario.")
        return None
    # --- END ADDED CHECK ---

    # Reconstruir columnas hora y dia_semana numéricas
    df["_hora"] = (df[horas_cols]
                   .idxmax(axis=1)
                   .str.extract(r"t_H(\d+)")[0]
                   .astype(int))
    df["_dia"]  = (df[dias_cols]
                   .idxmax(axis=1)
                   .map({c: i for i, c in enumerate(dias_cols)}))

    # ── Calcular pertenencia media por hora × día ────────────────────────
    # Para cada celda del grid, media de pertenencia de cada categoría
    grid_categoria = np.full((24, 7), "", dtype=object)
    grid_intensidad = np.zeros((24, 7))  # para el alpha del color

    for h in range(24):
        for d in range(7):
            mask = (df["_hora"] == h) & (df["_dia"] == d)
            sub = df[mask]
            if sub.empty:
                grid_categoria[h, d] = "v_Media"
                continue
            medias = {c: sub[c].mean() for c in cols_v}
            cat_dominante = max(medias, key=medias.get)
            grid_categoria[h, d] = cat_dominante
            grid_intensidad[h, d] = medias[cat_dominante]

    # ── Construir array RGBA para imshow ─────────────────────────────────
    import matplotlib.colors as mcolors
    rgba_grid = np.zeros((24, 7, 4))
    for h in range(24):
        for d in range(7):
            cat = grid_categoria[h, d]
            color_hex = COLORES_CONSECUENTE.get(cat, "#cccccc")
            r, g, b = mcolors.to_rgb(color_hex)
            # Alpha proporcional a la pertenencia media (mínimo 0.55)
            alpha = max(0.55, min(1.0, grid_intensidad[h, d] * 2))
            rgba_grid[h, d] = [r, g, b, alpha]

    # ── Plot ─────────────────────────────────────────────────────────────
    nombre_metrica_str = metrica.replace("_", " ").capitalize()


    fig, ax = plt.subplots(figsize=(9, 11))
    ax.imshow(rgba_grid, aspect="auto", origin="upper",
              extent=[-0.5, 6.5, 23.5, -0.5])

    # Ejes
    ax.set_xticks(range(7))
    ax.set_xticklabels(DIAS_SEMANA, fontsize=11)
    ax.set_yticks(range(24))
    ax.set_yticklabels([f"{h:02d}h" for h in range(24)], fontsize=9)
    ax.set_xlabel("Día de la semana", fontsize=11, labelpad=8)
    ax.set_ylabel("Hora del día", fontsize=11, labelpad=8)
    ax.set_title(
        f"Categoría dominante — {NOMBRE_DATASET}\n{nombre_metrica_str}",
        fontsize=13, fontweight="bold", pad=12
    )

    # Grid visual
    ax.set_xticks(np.arange(-0.5, 7, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 24, 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=0.6)
    ax.tick_params(which="minor", bottom=False, left=False)

    # Líneas separadoras de franjas horarias
    for franja_inicio in [0, 7, 14, 21]:
        ax.axhline(franja_inicio - 0.5, color="white", linewidth=1.8, alpha=0.7)

    # Etiquetas de franja en el margen derecho
    franjas = [(0, 6, "Madrugada"), (7, 13, "Mañana"),
               (14, 20, "Tarde"),   (21, 23, "Noche")]
    for ini, fin, nombre in franjas:
        ax.annotate(nombre, xy=(7.05, (ini + fin) / 2),
                    xycoords=("data", "data"),
                    fontsize=8, color="#555555", va="center",
                    annotation_clip=False)

    # Leyenda
    categorias_presentes = [c for c in orden if c in np.unique(grid_categoria)]
    patches = [
        mpatches.Patch(
            color=COLORES_CONSECUENTE[c],
            label=ETIQUETA_CONSECUENTE[c]
        )
        for c in categorias_presentes
    ]
    ax.legend(handles=patches, loc="lower center",
              bbox_to_anchor=(0.5, -0.13),
              ncol=len(patches), fontsize=9,
              frameon=True, edgecolor="#cccccc")

    plt.tight_layout()

    if guardar:
        ruta_img = f"data/{sensor}_{metrica}_heatmap.png"
        plt.savefig(ruta_img, dpi=150, bbox_inches="tight")
        print(f"Heatmap guardado en: {ruta_img}")

    plt.show()
    return fig




# ---------------------------------------------------------------------------
# 2. Gráfica de barras: top reglas ordenadas por lift
# ---------------------------------------------------------------------------

def grafica_barras_lift(df_reglas, sensor, metrica, top_n=20, guardar=True):
    """
    Barras horizontales con las top_n reglas ordenadas por lift descendente.
    Cada barra se colorea según su consecuente.
    """
    nombre_metrica_str = f"{METRICA}"

    top = df_reglas.nlargest(top_n, "lift").copy()
    top = top.iloc[::-1]  # invertir para que el mayor lift quede arriba

    etiquetas = [
        f"{row['antecedente']}  →  {ETIQUETA_CONSECUENTE.get(row['consecuente'], row['consecuente'])}"
        for _, row in top.iterrows()
    ]
    colores = [
        COLORES_CONSECUENTE.get(row["consecuente"], "#888888")
        for _, row in top.iterrows()
    ]

    fig, ax = plt.subplots(figsize=(11, max(5, len(top) * 0.45)))

    bars = ax.barh(range(len(top)), top["lift"], color=colores,
                   edgecolor="white", linewidth=0.5)

    # Etiqueta del valor de lift al final de cada barra
    for i, (bar, (_, row)) in enumerate(zip(bars, top.iterrows())):
        ax.text(bar.get_width() + 0.05, i,
                f"lift {row['lift']:.1f}  conf {int(row['confianza']*100)}%",
                va="center", fontsize=8, color="#444444")

    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(etiquetas, fontsize=8)
    ax.set_xlabel("Lift", fontsize=10)
    ax.set_title(
        f"Top {len(top)} reglas por lift — {sensor} | {nombre_metrica_str}",
        fontsize=11, fontweight="bold", pad=10
    )
    ax.axvline(1.0, color="#aaaaaa", linewidth=0.8, linestyle="--")
    ax.set_xlim(0, top["lift"].max() * 1.25)
    ax.grid(axis="x", alpha=0.3)

    # Leyenda de consecuentes presentes
    consecuentes_presentes = top["consecuente"].unique()
    patches = [
        plt.Rectangle((0, 0), 1, 1,
                       color=COLORES_CONSECUENTE.get(c, "#888888"),
                       label=ETIQUETA_CONSECUENTE.get(c, c))
        for c in sorted(consecuentes_presentes,
                        key=lambda x: list(ETIQUETA_CONSECUENTE.keys()).index(x)
                        if x in ETIQUETA_CONSECUENTE else 99)
    ]
    ax.legend(handles=patches, loc="lower right", fontsize=8,
              frameon=True, edgecolor="#cccccc", title="Categoría")

    plt.tight_layout()

    if guardar:
        ruta = f"data/{sensor}_{metrica}_barras_lift.png"
        plt.savefig(ruta, dpi=150, bbox_inches="tight")
        print(f"Guardado: {ruta}")

    plt.show()
    return fig


# ---------------------------------------------------------------------------
# 3. Tabla visual: resumen por consecuente
# ---------------------------------------------------------------------------

def grafica_tabla_consecuentes(df_reglas, sensor, metrica, guardar=True):
    """
    Tabla visual con una fila por consecuente:
    nº de reglas | confianza media | lift medio | lift máximo
    Coloreada por categoría.
    """
    nombre_metrica_str = f"{METRICA}"

    ORDEN = ["v_OutlierBajo","v_MuyBaja","v_Baja",
             "v_Media","v_Alta","v_MuyAlta","v_OutlierAlto"]

    resumen = (df_reglas
               .groupby("consecuente")
               .agg(
                   n_reglas   = ("lift", "count"),
                   conf_media = ("confianza", "mean"),
                   lift_medio = ("lift", "mean"),
                   lift_max   = ("lift", "max"),
               )
               .round(2))

    # Ordenar según escala semántica
    resumen = resumen.reindex(
        [c for c in ORDEN if c in resumen.index]
    )

    cols      = ["Categoría", "Nº reglas", "Conf. media", "Lift medio", "Lift máx."]
    filas     = []
    colores_f = []

    for cat, row in resumen.iterrows():
        filas.append([
            ETIQUETA_CONSECUENTE.get(cat, cat),
            int(row["n_reglas"]),
            f"{row['conf_media']*100:.0f} %",
            f"{row['lift_medio']:.2f}",
            f"{row['lift_max']:.2f}",
        ])
        colores_f.append(COLORES_CONSECUENTE.get(cat, "#cccccc"))

    fig, ax = plt.subplots(figsize=(9, 0.55 * len(filas) + 1.5))
    ax.axis("off")

    tabla = ax.table(
        cellText  = filas,
        colLabels = cols,
        cellLoc   = "center",
        loc       = "center",
    )
    tabla.auto_set_font_size(False)
    tabla.set_fontsize(10)
    tabla.scale(1, 1.6)

    # Colorear cabecera
    for j in range(len(cols)):
        tabla[0, j].set_facecolor("#333333")
        tabla[0, j].set_text_props(color="white", fontweight="bold")

    # Colorear filas por consecuente
    for i, color_hex in enumerate(colores_f):
        r, g, b = mcolors.to_rgb(color_hex)
        # Versión más clara para el fondo de la celda
        fondo = (r * 0.35 + 0.65, g * 0.35 + 0.65, b * 0.35 + 0.65)
        for j in range(len(cols)):
            tabla[i + 1, j].set_facecolor(fondo)

    ax.set_title(
        f"Resumen por categoría — Sensor {sensor} | {nombre_metrica_str}",
        fontsize=11, fontweight="bold", pad=12
    )
    plt.tight_layout()

    if guardar:
        ruta = f"data/{sensor}_{metrica}_tabla_consecuentes.png"
        plt.savefig(ruta, dpi=150, bbox_inches="tight")
        print(f"Guardado: {ruta}")

    plt.show()
    return fig


# ---------------------------------------------------------------------------
# Ejecución — llama a las 3 funciones con SENSOR y METRICA de la celda 2
# ---------------------------------------------------------------------------

print(f"Generando visualizaciones para Sensor {NOMBRE_DATASET} | {METRICA}...\n")
generar_heatmap(NOMBRE_DATASET, METRICA, guardar=True)
grafica_barras_lift(df_reglas, NOMBRE_DATASET, METRICA)
grafica_tabla_consecuentes(df_reglas, NOMBRE_DATASET, METRICA)
print("\nVisualizaciones completadas.")


In [ ]:
# ---------------------------------------------------------------------------
# 7. Ejecución y visualización
# ---------------------------------------------------------------------------

resumen = generar_resumen(
    df_reglas,
    dataset=NOMBRE_DATASET,
    metrica=METRICA,
    min_reglas_grupo=MIN_REGLAS_GRUPO,
)

print(resumen)


In [ ]:
# ---------------------------------------------------------------------------
# 8. Guardado del resumen en fichero .PDF
# ---------------------------------------------------------------------------

with open(FICHERO_RESUMEN, "w", encoding="utf-8") as f:
    f.write(resumen)

print(f"Resumen guardado en: {FICHERO_RESUMEN}")


In [ ]:
def test_coherencia_franjas(df_reglas, mapa_horario):
    """
    Verifica que el nivel narrado en el párrafo coloquial coincide con el nivel
    dominante de las reglas que aparecen en la tabla por franja.
    """
    NIVEL_PESO = {
        "v_OutlierAlto": 7, "v_MuyAlta": 6, "v_Alta": 5,
        "v_Media": 4,
        "v_Baja": 3, "v_MuyBaja": 2, "v_OutlierBajo": 1,
    }
    HORAS_FRANJA = {
        "t_Madrugada": list(range(0, 7)),
        "t_Mañana":    list(range(7, 14)),
        "t_Tarde":     list(range(14, 21)),
        "t_Noche":     list(range(21, 24)),
    }
    inconsistencias = []
    for franja, horas in HORAS_FRANJA.items():
        # Nivel del párrafo coloquial: el de mayor peso entre las horas mapeadas
        niveles_parrafo = [mapa_horario.get(h) for h in horas if h in mapa_horario]
        if not niveles_parrafo:
            continue
        nivel_parrafo = max(niveles_parrafo, key=lambda n: NIVEL_PESO.get(n, 0))

        # Nivel de la tabla: el consecuente del lift máximo entre las reglas con esa franja
        patron = "|".join(f"t_H{h:02d}" for h in horas)
        sub = df_reglas[df_reglas["antecedente"].str.contains(patron, regex=True)]
        if sub.empty:
            continue
        # Tomamos el más representativo por peso x lift
        votos = {}
        for _, row in sub.iterrows():
            n = row["consecuente"]
            votos[n] = votos.get(n, 0) + row["lift"] * row["confianza"]
        nivel_tabla = max(votos, key=votos.get) if votos else None

        if nivel_parrafo != nivel_tabla:
            inconsistencias.append({
                "franja": franja,
                "nivel_parrafo": nivel_parrafo,
                "nivel_tabla": nivel_tabla,
            })

    if inconsistencias:
        print("⚠️  INCONSISTENCIAS detectadas:")
        for i in inconsistencias:
            print(f"  {i['franja']}: párrafo dice {i['nivel_parrafo']}, "
                  f"tabla dice {i['nivel_tabla']}")
    else:
        print("✓ Párrafo y tabla coherentes en todas las franjas.")
    return inconsistencias

# Ejemplo de uso (descomentar cuando df_reglas y mapa_horario estén disponibles)
# test_coherencia_franjas(df_reglas, mapa_horario)
